In [1]:
import glob
import numpy as np
from pytoast.ocean.adv import ADV
from pytoast.utils.io_utils import results_to_dataset

In [2]:
# Files from test directory
files = glob.glob("../py-tests/src/ocean/adv/testdata/mat_list/*.mat")

# Deployment specifics
water_depth = 13
depth_array = np.linspace(1.8, 7.2, 6)

# pytoast_name: source_name pairs
name_map = {"u1": "E", "u2": "N", "u3": "w", "p": "P2", "time": "dn"}

# Initializing ADV object
adv = ADV(files, name_map, fs=32, z=depth_array, z_convention="depth", source_coords="enu", orientation="up", water_depth=water_depth)

# Setting pre-processing options
pre_opts = {
    "despike": {"method": "goring_nikora"},
    "rotate": {
        "flow_rotation": "align_streamwise",
    },
}
adv.set_preprocess_opts(pre_opts)

In [3]:
# Looping through bursts and computing various statistics.
# Storing output dictionaries in results list to export to netCDF
results = []
time_array = []
freq = None
for ii in range(adv.n_bursts):
    # Loading burst which contains u1, u2, u3, pressure, time
    burst = adv.load_burst(ii)

    # Extracting time and calculating derived statistics
    time_start = burst["time"][0, 0]
    cov = adv.covariance(burst, method="benilov")
    wave_stats = adv.directional_wave_statistics(burst)
    eps = adv.dissipation(burst, f_low=1.2, f_high=10)

    # Grabbing frequency vector from first burst
    if freq is None:
        freq = wave_stats["f"][0]
    wave_stats.pop("f", None)

    # Storing everything in one dictionary
    burst_output = {}
    burst_output.update(cov)
    burst_output.update(wave_stats)
    burst_output.update(eps)

    results.append(burst_output)
    time_array.append(time_start)

In [4]:
ds = results_to_dataset(results, burst_times=np.array(time_array), z=adv.z, freq=freq, attrs={"instrument": "ADV", "z_convention": "depth"})

/Users/ea-gegan/Documents/gitrepos/pytoast/src/pytoast/utils/io_utils.py:104: ComplexWarning: Casting complex values to real discards the imaginary part
  v_arr = np.asarray(v, dtype=float)
